
# Dimensionality Reduction Techniques

In modern data science, we often face the "Curse of Dimensionality": datasets with hundreds or thousands of features (columns).

-   **Visualization**: We cannot plot 100 dimensions.
-   **Computation**: Algorithms become slow.
-   **Overfitting**: With too many features, models find spurious patterns.

**Dimensionality Reduction** is the process of compressing this data into fewer features while keeping the important "signal". Common techniques include:

-   **Principal Component Analysis (PCA)**: A linear technique that projects data onto directions of maximum variance.
-   **Autoencoders**: Neural networks that learn compressed representations through encoding and decoding.
-   **t-SNE / UMAP**: Non-linear techniques primarily used for visualization in 2D/3D.

## Principal Component Analysis (PCA)

PCA is the most popular linear technique. It doesn't just delete columns; it finds a **new coordinate system** for the data.

1.  It finds the direction of **Maximum Variance** (the "spread" of the data). This becomes **Principal Component 1 (PC1)**.
2.  It finds a second direction perpendicular (orthogonal) to the first that captures the remaining variance. This is **PC2**.
3.  And so on.

### Why "Variance" Matters

Information $\approx$ Variance.

-   If a feature has zero variance (all values are 5), it carries zero information.
-   PCA keeps the directions with high variance (signal) and discards directions with low variance (noise).

### Mathematical Formulation

Given a dataset $X \in \mathbb{R}^{n \times p}$ with $n$ samples and $p$ features, PCA involves:

1.  **Standardization**: Center the data by subtracting the mean: $$X' = X - \bar{X}$$
2.  **Covariance Matrix**: Compute $C = \frac{1}{n-1} X'^T X'$
3.  **Eigenvalue Decomposition**: $C = V \Lambda V^T$, where $V$ contains eigenvectors and $\Lambda$ contains eigenvalues.
4.  **Project**: Select top $k$ eigenvectors and project: $Z = X' W_k$

### Advantages and Disadvantages

-   **Advantages**:
    -   Reduces dimensionality while preserving variance.
    -   Helps in noise reduction by discarding less significant components.
    -   Can improve model performance by reducing overfitting.
-   **Disadvantages**:
    -   Assumes linear relationships between features.
    -   Sensitive to feature scale; requires standardization.
    -   Cannot capture complex, non-linear relationships.

### Evaluating PCA

-   **Explained Variance Ratio**: How much variance each component captures.
-   **Scree Plot**: Visualizes variance per component; the "elbow" helps choose $k$.
-   **Reconstruction Error**: How well the reduced data reconstructs the original.

## Practical Demonstration: PCA from Scratch

We will generate a 2D dataset that is highly correlated (shaped like a cigar) and use PCA to rotate it.

### Generate Correlated Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
np.random.seed(42)

# Generate 2 features with high correlation
mean = [0, 0]
cov = [[1, 0.9], [0.9, 1]]  # High covariance (0.9)
X = np.random.multivariate_normal(mean, cov, 200)

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], alpha=0.6, color='teal')
plt.title("Original Correlated Data")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.axis('equal')
plt.grid(True)
plt.show()

### Apply PCA

We use scikit-learn's `PCA` to find the axes.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# The Components (The new axes directions)
print("Principal Axes (Eigenvectors):\n", pca.components_)

# The Variance Explained (Eigenvalues)
print("\nExplained Variance Ratio:", pca.explained_variance_ratio_)

**Observation**: PC1 explains ~95% of the variance. PC2 explains only ~5%. This means we can drop PC2 and lose very little information.

### Visualizing the Rotation

Let's plot the data in the new "PCA Space". Notice how the correlation is gone! The blob is now aligned with the axes.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, color='purple')
plt.title("Data in PCA Space (Rotated)")
plt.xlabel("Principal Component 1 (High Variance)")
plt.ylabel("Principal Component 2 (Low Variance)")
plt.axis('equal')
plt.grid(True)
plt.show()

### Dimensionality Reduction (Compression)

Now we keep only PC1. We effectively flatten the data from 2D to 1D. We can then **inverse transform** it back to 2D to see what was lost.

In [ ]:
# Drop PC2 by setting it to 0
X_pca_1d = X_pca.copy()
X_pca_1d[:, 1] = 0

# Reconstruct original data from just 1 component
X_reconstructed = pca.inverse_transform(X_pca_1d)

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], alpha=0.3, label='Original')
plt.scatter(X_reconstructed[:, 0], X_reconstructed[:, 1], alpha=0.8, color='red', label='Reconstructed (1D)')
plt.title("Compression: 2D -> 1D")
plt.legend()
plt.axis('equal')
plt.show()

**Result**: The red dots lie perfectly on a line. We lost the "width" of the blob (noise), but kept the "length" (signal).

## Autoencoders (Non-Linear Reduction)

PCA is linear. It can only rotate and stretch. If your data lies on a curve (like a Swiss Roll), PCA fails.

**Autoencoders** are Neural Networks that can learn **curved** compressions. They consist of:

-   **Encoder**: Compresses input ($X$) into a bottleneck representation ($Z$).
-   **Decoder**: Reconstructs ($X'$) from the bottleneck.
-   If $X \approx X'$, then $Z$ is a good compressed representation.

The network is trained to minimize reconstruction error: $$\mathcal{L}(X, \hat{X}) = \frac{1}{n} \sum_{i=1}^{n} \|X_i - \hat{X}_i\|^2$$

Autoencoders excel at:

-   Learning non-linear manifolds
-   Denoising (reconstructing clean data from noisy inputs)
-   Feature extraction for downstream tasks

## t-SNE (Visualizing High Dimensions)

t-SNE (t-Distributed Stochastic Neighbor Embedding) is strictly for **visualization**. It preserves local neighbors: "If points A and B are close in 100 dimensions, keep them close in 2D."

**Warning**: Do not use t-SNE for clustering or feature engineering. Distances in t-SNE are distorted.

## Exercises

We will use the **Digits Dataset** (8x8 pixel images of handwritten numbers). This is a 64-dimensional dataset ($8 \times 8 = 64$). We will crush it to 2 dimensions.

### Load and Inspect

### PCA Reduction

-   Reduce from 64D to 2D using PCA.
-   Plot the 2D result. Color the points by the digit label (0-9).

**Observation**: You should see that visually similar digits (like 4 and 9, or 3 and 8) tend to cluster together, while very different ones separate.

### The Scree Plot (How many components?)

How much information did we lose by going to 2D?

-   Fit PCA with `n_components=64` (Full).
-   Plot `cumsum(explained_variance_ratio_)`.
-   Find how many components are needed for 95% variance.

**Result**: Usually around 20-30 components are enough to capture 95% of the image data, reducing the size by half without losing quality!

### Correlated Features Exercise

Create a synthetic dataset with 4 highly correlated features and apply PCA. **Question**: Given the high correlations, how many components capture most of the variance?

## Summary

1.  **Curse of Dimensionality**: High dimensions cause computational and statistical problems.
2.  **PCA**: The gold standard. Rotates data to align with variance. Linear.
3.  **Components**: We choose $k$ components to balance compression vs. information loss (using a Scree Plot).
4.  **Autoencoders**: Neural networks for non-linear dimensionality reduction.
5.  **t-SNE/UMAP**: Better for visualizing complex clusters, but tricky to interpret.